# Evaluate using AI as Judge Quality Evaluators with Azure AI Evaluation SDK

## Objective

This tutorial provides a step-by-step guide on how to evaluate prompts against variety of model endpoints deployed on Azure AI Platform or non Azure AI platforms. 

This guide uses Python Class as an application target which is passed to Evaluate API provided by Azure AI Evaluation SDK to evaluate results generated by LLM models against provided prompts. 

This tutorial uses the following Azure AI services:

- [azure-ai-evaluation](https://learn.microsoft.com/en-us/azure/ai-studio/how-to/develop/evaluate-sdk)

## Time

You should expect to spend 30 minutes running this sample. 

## About this example

This example demonstrates evaluating model endpoints responses against provided prompts using azure-ai-evaluation

## Before you begin

### Installation

Install the following packages required to execute this notebook. 

In [1]:
# %pip install azure-ai-evaluation
# %pip install promptflow-azure
# %pip install azure-identity
# %pip install --upgrade openai

### Parameters and imports

In [2]:
from pprint import pprint
import pandas as pd
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
load_dotenv()

True

## Target Application

We will use Evaluate API provided by Prompt Flow SDK. It requires a target Application or python Function, which handles a call to LLMs and retrieve responses. 

In the notebook, we will use an Application Target `ModelEndpoints` to get answers from multiple model endpoints against provided question aka prompts. 

This application target requires list of model endpoints and their authentication keys. For simplicity, we have provided them in the `env_var` variable which is passed into init() function of `ModelEndpoints`.


Please provide Azure AI Project details so that traces and eval results are pushing in the project in Azure AI Studio.

In [3]:
# azure_ai_project = {
#     "subscription_id": "<your-subscription-id>",
#     "resource_group_name": "<your-resource-group-name>",
#     "project_name": "<your-project-name>",
# }

# azure_openai_api_version = "<api version>"
# azure_openai_deployment = "<your-deployment>"
# azure_openai_endpoint = "<your-endpoint>"
# azure_openai_api_key

## Data

Following code reads Json file "data.jsonl" which contains inputs to the Application Target function. It provides question, context and ground truth on each line. 

In [4]:
df = pd.read_json("data.jsonl", lines=True)
print(df.head())

                                           query  \
0                 What is the capital of France?   
1             Which tent is the most waterproof?   
2           Which camping table is the lightest?   
3  How much does TrailWalker Hiking Shoes cost?    

                                             context  \
0                   France is the country in Europe.   
1  #TrailMaster X4 Tent, price $250,## BrandOutdo...   
2  #BaseCamp Folding Table, price $60,## BrandCam...   
3  #TrailWalker Hiking Shoes, price $110## BrandT...   

                                        ground_truth  
0                                              Paris  
1  The TrailMaster X4 tent has a rainfly waterpro...  
2  The BaseCamp Folding Table has a weight of 15 lbs  
3    The TrailWalker Hiking Shoes are priced at $110  


## Configuration
To use Relevance and Cohenrence Evaluator, we will Azure Open AI model details as a Judge that can be passed as model config.

In [5]:
import os
from azure.ai.evaluation import AzureOpenAIModelConfiguration

azure_ai_project = os.environ.get("AZURE_AI_PROJECT")

model_config = AzureOpenAIModelConfiguration(
    azure_deployment=os.environ.get("AZURE_OPENAI_DEPLOYMENT"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
)

print(os.environ.get("AZURE_OPENAI_API_KEY"))
print(os.environ.get("AZURE_OPENAI_ENDPOINT"))
print(os.environ.get("AZURE_OPENAI_API_VERSION"))
print(os.environ.get("AZURE_OPENAI_DEPLOYMENT"))


1ecgTXR8OPaoeSSgW9Q3uC9yKwpmaNNkFFyHk69yKirSZPh7UacWJQQJ99BDAC8vTInXJ3w3AAAAACOGotJ7
https://build-2025-fdp-test-account1.openai.azure.com/openai
2025-05-15-preview
gpt-4.1


In [6]:
from model_endpoint import ModelEndpoint

model = ModelEndpoint(model_config)
model(query="Test")

{'azure_deployment': 'gpt-4.1', 'azure_endpoint': 'https://build-2025-fdp-test-account1.openai.azure.com/openai', 'api_version': '2025-05-15-preview', 'api_key': '1ecgTXR8OPaoeSSgW9Q3uC9yKwpmaNNkFFyHk69yKirSZPh7UacWJQQJ99BDAC8vTInXJ3w3AAAAACOGotJ7'}
Azure OpenAI model endpoint
{'azure_deployment': 'gpt-4.1', 'azure_endpoint': 'https://build-2025-fdp-test-account1.openai.azure.com/openai', 'api_version': '2025-05-15-preview', 'api_key': '1ecgTXR8OPaoeSSgW9Q3uC9yKwpmaNNkFFyHk69yKirSZPh7UacWJQQJ99BDAC8vTInXJ3w3AAAAACOGotJ7'}


NotFoundError: Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}

## Run the evaluation

The Following code runs Evaluate API and uses Content Safety, Relevance and Coherence Evaluator to evaluate results from different models.

The following are the few parameters required by Evaluate API. 

+   Data file (Prompts): It represents data file 'data.jsonl' in JSON format. Each line contains question, context and ground truth for evaluators.     

+   Application Target: It is name of python class which can route the calls to specific model endpoints using model name in conditional logic.  

+   Model Name: It is an identifier of model so that custom code in the App Target class can identify the model type and call respective LLM model using endpoint URL and auth key.  

+   Evaluators: List of evaluators is provided, to evaluate given prompts (questions) as input and output (answers) from LLM models. 

In [ ]:
import pathlib

from azure.ai.evaluation import evaluate
from azure.ai.evaluation import (
    ContentSafetyEvaluator,
    RelevanceEvaluator,
    CoherenceEvaluator,
    GroundednessEvaluator,
    FluencyEvaluator,
    SimilarityEvaluator,
)
from model_endpoint import ModelEndpoint


content_safety_evaluator = ContentSafetyEvaluator(
    azure_ai_project=azure_ai_project, credential=DefaultAzureCredential()
)
relevance_evaluator = RelevanceEvaluator(model_config)
coherence_evaluator = CoherenceEvaluator(model_config)
groundedness_evaluator = GroundednessEvaluator(model_config)
fluency_evaluator = FluencyEvaluator(model_config)
similarity_evaluator = SimilarityEvaluator(model_config)

path = str(pathlib.Path(pathlib.Path.cwd())) + "/data.jsonl"

results = evaluate(
    evaluation_name="Eval-Run-" + "-" + model_config["azure_deployment"].title(),
    data=path,
    target=ModelEndpoint(model_config),
    evaluators={
        "content_safety": content_safety_evaluator,
        "coherence": coherence_evaluator,
        "relevance": relevance_evaluator,
        "groundedness": groundedness_evaluator,
        "fluency": fluency_evaluator,
        "similarity": similarity_evaluator,
    },
    evaluator_config={
        "content_safety": {"column_mapping": {"query": "${data.query}", "response": "${target.response}"}},
        "coherence": {"column_mapping": {"response": "${target.response}", "query": "${data.query}"}},
        "relevance": {
            "column_mapping": {"response": "${target.response}", "context": "${data.context}", "query": "${data.query}"}
        },
        "groundedness": {
            "column_mapping": {
                "response": "${target.response}",
                "context": "${data.context}",
                "query": "${data.query}",
            }
        },
        "fluency": {
            "column_mapping": {"response": "${target.response}", "context": "${data.context}", "query": "${data.query}"}
        },
        "similarity": {
            "column_mapping": {"response": "${target.response}", "context": "${data.context}", "query": "${data.query}"}
        },
    },
)

View the results

In [ ]:
pprint(results)

In [ ]:
pd.DataFrame(results["rows"])